# 3 · make — Helico structure-accuracy data

Aggregates the per-target GDT-TS that [Open-Athena/helico](https://github.com/Open-Athena/helico)'s
exp14 published, over the same 333 FoldBench monomers #245 scores. Helico folds each protein from
a contact map; the arms differ only in where the contacts came from, so `Helico, no contacts` and
`Helico + true contacts` bracket what contact conditioning can do at all.

Restricted to targets **every** arm scored, so the arms are compared on one protein set rather
than on whichever ones each of them happened to finish.

CPU only; read anonymously from the public bucket.

In [1]:
# Run from anywhere: figlib lives next to this notebook.
import sys
from pathlib import Path

HERE = Path.cwd() if (Path.cwd() / "figlib.py").exists() else Path("notebooks/figures")
sys.path.insert(0, str(HERE.resolve()))
import figlib

In [2]:
# --- parameters -------------------------------------------------------------------------------
DATASET = "3_gdt_ts"
METRIC = "gdt_ts"      # also available per target: lddt, tm_score, rmsd
BOOTSTRAP_DRAWS = 2000
BOOTSTRAP_SEED = 0
CLASSES = {"natural": 0, "designed": 1}   # name -> the `designed` flag it selects
PARAMETERS = dict(metric=METRIC, classes=list(CLASSES),
                  bootstrap_draws=BOOTSTRAP_DRAWS, bootstrap_seed=BOOTSTRAP_SEED)
PARAMETERS

{'metric': 'gdt_ts',
 'classes': ['natural', 'designed'],
 'bootstrap_draws': 2000,
 'bootstrap_seed': 0}

In [3]:
import io

import pandas as pd

inputs = figlib.Inputs()
frame = pd.read_csv(io.BytesIO(inputs.fetch(f"{figlib.HELICO}/scores/per_target.csv")))
scored = frame[frame.status == "ok"]
complete = scored.groupby("target_id").arm.nunique()
keep = set(complete[complete == scored.arm.nunique()].index)
dropped = sorted(set(scored.target_id) - keep)
per_target = scored[scored.target_id.isin(keep)]
print(f"{scored.arm.nunique()} arms · {len(keep)} targets scored by all of them"
      + (f" · {len(dropped)} dropped: {dropped}" if dropped else ""))

11 arms · 324 targets scored by all of them · 9 dropped: ['7pv5_A', '7t9r_A', '8es6_A', '8por_A', '8q79_A', '8qle_A', '8ra0_A', '8u0i_A', '9b8e_A']


In [4]:
rows = []
for class_name, designed in CLASSES.items():
    subset = per_target[per_target.designed == designed]
    for arm, group in subset.groupby("arm"):
        mean, low, high = figlib.bootstrap_mean(group[METRIC].values, BOOTSTRAP_DRAWS,
                                                BOOTSTRAP_SEED)
        rows.append(dict(protein_class=class_name, arm=arm, n=len(group),
                         value=mean, ci_low=low, ci_high=high))

summary = pd.DataFrame(rows).sort_values(["protein_class", "value"], ascending=[True, False])
print(summary.to_string(index=False, float_format=lambda v: f"{v:.3f}"))

protein_class                    arm   n  value  ci_low  ci_high
     designed               esmfold2  19  0.934   0.908    0.956
     designed                 oracle  19  0.920   0.886    0.947
     designed protenix_v2_single_seq  19  0.892   0.851    0.928
     designed                   v2ss  19  0.876   0.830    0.917
     designed        protenix_v2_msa  19  0.860   0.771    0.923
     designed                    off  19  0.859   0.793    0.913
     designed                  mf_L5  19  0.856   0.801    0.903
     designed                  v2msa  19  0.850   0.760    0.916
     designed                esmfold  19  0.797   0.637    0.924
     designed                  mf_L2  19  0.764   0.649    0.860
     designed                   mf_L  19  0.761   0.647    0.861
      natural                 oracle 305  0.893   0.878    0.906
      natural        protenix_v2_msa 305  0.868   0.850    0.887
      natural                  v2msa 305  0.827   0.806    0.846
      natural            

In [5]:
figlib.write_dataset(
    DATASET,
    notebook="3_make_gdt_ts_data.ipynb",
    parameters=PARAMETERS,
    inputs=inputs,
    files={
        "summary.csv": lambda path: summary.to_csv(path, index=False),
        "per_target.csv": lambda path: per_target.to_csv(path, index=False),
    },
    extra={
        "metric": {"name": METRIC, "source": "Open-Athena/helico exp14"},
        "arms_dropped_targets": dropped,
        "arms": sorted(per_target.arm.unique()),
    })

wrote 2 file(s) + metadata.json to /home/bizon/git/MarinFold/.claude/worktrees/evals-exploration-notebook-cba1c7/notebooks/figures/data/3_gdt_ts
   summary.csv                           1,749 B  8916259d7ac3
   per_target.csv                      484,463 B  e3a4a00280be


PosixPath('/home/bizon/git/MarinFold/.claude/worktrees/evals-exploration-notebook-cba1c7/notebooks/figures/data/3_gdt_ts')